In [366]:
import pandas as pd
from pandas._libs.tslibs.offsets import CustomBusinessDay

###### Esse notebook é de um projeto ainda em desenvolvimento. Ele é atualizado uma vez antes de cada release. Se você clonou o repositório a partir do commit mais recente, algumas informações podem estar desatualizadas. ######

# 1. Sobre os dados

Por enquanto, esse projeto apenas analisa dois indicadores econômicos: o CDI e o IPCA. Todos os dados são armazenados

### 1.1.1 CDI
O Certificado de Depósito Interbancário (CDI) é um título emitido por bancos e instituições financeiras para permitir empréstimos entre si, e terminar o dia com o caixa positivo. Ele também é uma referência para várias outras operações financeiras no Brasil. Ele é relevante para o investidor comum quando se considera a atrelação entre o CDI, e investimentos de renda fixa como CDBs. E também porque, em geral, um investimento que rende pelo menos tanto quanto o CDI é considerado um bom investimento.

Esse é o esquema de como o CDI é armazenado.


In [367]:
# Você pode consultar ss esquemas das planilhas em src/storage/processed/schema.py
def cdi_schema() -> pd.DataFrame:
    df = pd.DataFrame({
        "date": [],
        "cdi_annual_rate": [],
        "cdi_monthly_rate": [],
        "cdi_accumulated_ytd_rate": [],
        "cdi_12m_rate": [],
    })
    return df

Aqui, vale a pena falar alguns pontos sobre a natureza do CDI como índice e taxa. Como você pode ver, o CDI não é um valor só. Aqui, armazenamos a taxa anual (a.a) e a taxa mensal. Que são os acumulos do CDI em um ano e em um mês. O CDI próprio, na verdade, tem prazo de apenas um dia útil. E o índice do CDI é o valor absoluto do CDI durante esse dia. Mas em nosso projeto armazenamos a taxa mensal do CDI, que é a variação do CDI (ou, o CDI acumulado) em um mês.

No notebook 02 falaremos sobre um cuidado importante que deveremos tomar por causa dessa escolha. Por enquanto, vamos rodar a Pipeline e ver os resultados.

Utilizaremos esse comando no terminal:
```bash
py main.py --mode yearly --year 2024 --clear-data
```

Isso vai popular a planilha em /data/processed, mas note que dados brutos também são armazenados em /data/raw.
Vamos abrir o json dos dados brutos para a taxa mensal do CDI em Janeiro de 2024:
```json
{
  "reference_date": "2024-01",
  "type": "cdi",
  "value": {
    "monthly": 0.0097,
    "yearly": 0.1165
  }
}
```
0.0097 equivale a 0,97%. Como padrão, esse projeto sempre armazenará os valores em formato decimal para facilitar cálculos posteriores. A transformação de volta para a porcentagem (e a substituição de ponto para vírgula) deve ser feita apenas nas visualizações.

Bem, mas afinal, como que chegamos nesses números?
De novo, o índice do CDI se refere a apenas um dia. A taxa mensal é a mensuração do crescimento relativo do CDI entre o último dia útil do mês anterior, e o último dia útil do mês calculado. Nesse cálculo, a seguinte fórmula é utilizada.

$$\large\frac{Índice_{final}}{Índice_{inicial}} - 1$$

Verificando a série histórica do DI fornecida pela B3, o índice do CDI em 29-12-2023 estava em **42805,60**, e em 31-01-2024 o índice estava em **43219,40**. Substituindo esses valores na fórmula, temos:

In [368]:
def calc_cdi_monthly_rate(start_index: float, end_index: float) -> float:
    return end_index / start_index - 1

jan_monthly_rate = calc_cdi_monthly_rate(start_index=42805.60, end_index=43219.40)

print(f"Taxa mensal: {jan_monthly_rate}")
print(f"Taxa mensal arredondada: {round(jan_monthly_rate, 4)}")


Taxa mensal: 0.009666959463247915
Taxa mensal arredondada: 0.0097


A taxa anual é calculada da mesma maneira, porém a taxa é anualizada. Na prática, ela conta sobre como o resultado anual do CDI seria se ele rendesse no mesmo ritmo do mês calculado. Se utiliza essa fórmula:

$$\huge(1 + Taxa_{mensal})^{\frac{252}{Ndu}} - 1$$

Onde Ndu é o número de dias úteis no mês calculado.

Criar uma função para calcular a taxa anualizada no python é um pouco mais complicado, porque o calendário de feriados que devemos utilizar é diferente do que o pandas utiliza na função bdate_rate. Mas, para demonstrar como esse cálculo seria, carregamos uma lista personalizada dos feriados de Janeiro de 2024 (retirada no [Calendário da FEBRABAM](https://feriadosbancarios.febraban.org.br/)), e utilizamos esse número na conta.

In [369]:
jan_holidays = ["2024-01-01"]

jan_bday_febraban = CustomBusinessDay(holidays=jan_holidays)

start = pd.Timestamp(2024, 1, 1)
end = start + pd.offsets.MonthEnd(0)

bdays_no = len(pd.date_range(start, end, freq=jan_bday_febraban))

def annualize_cdi(monthly_rate: float, bdays_no: int) -> float:
    return (1 + monthly_rate)**(252 / bdays_no) - 1

annalized_rate = annualize_cdi(monthly_rate=jan_monthly_rate, bdays_no=bdays_no)
print(f"Taxa anualizada: {annalized_rate}")
print(f"Taxa anualizada arredondada: {round(annalized_rate, 4)}")

Taxa anualizada: 0.11650004967146854
Taxa anualizada arredondada: 0.1165
0.11650004967146854


Perfeito, agora resta falar sobre os números calculados pela Pipeline. Em específico, o cdi_accumulated_ytd_rate e o cdi_12m_rate. Esses dois campos são preenchidos com a conta de acúmulos.hh



### 1.1.2 IPCA
O Índice Nacional de Preços ao Consumidor Amplo (IPCA) é o índice oficial de inflação do Brasil. Ele é calculado mensalmente pelo IBGE, e reflete a variação de preços de um conjunto de bens e serviços consumidos pelas famílias brasileiras. Ele é relevante para o investidor comum porque a inflação tem um impacto direto no poder de compra do dinheiro.

Esse é o esquema de como o IPCA é armazenado.

In [370]:
# Você pode consultar ss esquemas das planilhas em src/storage/processed/schema.py
def ipca_schema() -> pd.DataFrame:
    df = pd.DataFrame({
        "date": [],
        "ipca_monthly_rate": [],
        "ipca_accumulated_ytd_rate": [],
        "ipca_12m_rate": [],
    })
    return df

Seus dados brutos também são armazenados de uma maneira semelhante ao do CDI.